# Hallmark Python Demo: init, info, add, commit, checkout, status, and clone


## Setup


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from hallmark import Repo
from hallmark.downloader import download_remote_data
from hallmark.error import DestinationExistsError


In [2]:
%%bash
rm -rf repo1
rm -rf hallmark-demo-clone
mkdir repo1


## 1. Initialize the hallmark repo


In [3]:
repo1 = Repo.init("repo1")


### Repository info

The Python API exposes the same local hallmark paths directly on the `Repo` object.

In [4]:
print(f'dot-hallmark repo: "demo/{Path(repo1.dothm.path).relative_to(Path.cwd())}"')
print(f'hallmark worktree: "demo/{Path(repo1.worktree).relative_to(Path.cwd())}"')

dot-hallmark repo: "demo/repo1/.hm"
hallmark worktree: "demo/repo1"


In [5]:
%%bash
echo "=== .hm directory ==="
ls repo1/.hm/
echo ""
echo "=== initial config.yml ==="
cat repo1/.hm/config.yml
echo ""
echo "=== objects stored so far ==="
find repo1/.hm/objects -type f 2>/dev/null | wc -l


=== .hm directory ===
config.yml
data.tsv
meta.yml
README.md

=== initial config.yml ===
# Edit this file only if your branch needs regex substitutions.
# For simple names, you can just run: hallmark add "a{a}_i{i}.h5"
data:
  -
    # fmt: "{release}_{source}_{year}_{doy:03d}_{band}.uvfits"
    encoding:
      # aspin: m([0-9]+(\.[0-9]+)?|\.[0-9]+)
remote:
  # name: origin
  # url: https://example.com/path/to/data/

=== objects stored so far ===
       0


## 2. Create data files on `main`


In [6]:
%%bash
pushd repo1
for f in a{0,0.75}_i{0,30,60,90,120}.h5; do echo "$f" > "$f"; done
echo "Files in repo1/:"
ls *.h5
popd


~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo/repo1 ~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo
Files in repo1/:
a0_i0.h5
a0_i120.h5
a0_i30.h5
a0_i60.h5
a0_i90.h5
a0.75_i0.h5
a0.75_i120.h5
a0.75_i30.h5
a0.75_i60.h5
a0.75_i90.h5
~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo


## 3. Add files and commit on `main`


In [7]:
pf = repo1.add("a{a}_i{i}.h5")
pf

,path,a,i
0,a0.75_i0.h5,0.75,0.0
1,a0.75_i120.h5,0.75,120.0
2,a0.75_i30.h5,0.75,30.0
3,a0.75_i60.h5,0.75,60.0
4,a0.75_i90.h5,0.75,90.0
5,a0_i0.h5,0.00,0.0
6,a0_i120.h5,0.00,120.0
7,a0_i30.h5,0.00,30.0
8,a0_i60.h5,0.00,60.0
9,a0_i90.h5,0.00,90.0


In [8]:
%%bash
echo "=== config.yml after add ==="
cat repo1/.hm/config.yml
echo ""
echo "=== data.tsv after add ==="
cat repo1/.hm/data.tsv


=== config.yml after add ===
data:
- fmt: a{a}_i{i}.h5
  encoding: null
remote: null

=== data.tsv after add ===
sha1	a	i
f656d419104e7af783c119186a4d81b15563310f	0.75	0
ec4dae9f0b8f69ac42f0c290fc84e0d6f1bec6cd	0.75	120
890b2ea509575be9bc74ad515731a2562cd7406e	0.75	30
d9cfcdd0b44d81c10f88e126d5b1eea4914259a0	0.75	60
ed2cfc6bb157a42729d3c6f45b228b432edec4be	0.75	90
18682593a012503b71935017dbc044665254a23c	0	0
1c057a4885fd04379ee1217375720665713c902f	0	120
829b87d845b367a9cc87df6ff510d1bdb7444aca	0	30
075a30e0a9c4dddd92447f463219d4c2a842883d	0	60
de40f4983d86342b03809bfcc09958f43f359b89	0	90


In [9]:
repo1.commit("add main dataset")


True

In [10]:
%%bash
echo "Objects stored after commit:"
find repo1/.hm/objects -type f | wc -l


Objects stored after commit:
      10


### Commit log


In [11]:
print(repo1.log())


commit 01b30149cdbd0380739e3a22c8ebbcdd33c2618d
Author: rohinsant <rohinsant@dhcp-10-134-174-6.uawifi.arizona.edu>
Date:   Thu Apr 23 17:26:52 2026 -0700

    add main dataset

commit a45f3726de0b3b93585e765b1f5d50c58b7875fb
Author: rohinsant <rohinsant@dhcp-10-134-174-6.uawifi.arizona.edu>
Date:   Thu Apr 23 17:26:52 2026 -0700

    Initial commit: local `.hm` repository


### Clean status


In [12]:
repo1.status()


{'branch': 'main',
 'staged': {'added': [], 'modified': [], 'deleted': []},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

### Current add modes

- `repo.add("a{a}_i{i}.h5")` parses files using the branch fmt and updates `config.yml`.
- `repo.add(".")` rebuilds `data.tsv` from files that currently exist using the fmt already set in `config.yml`.
- `repo.set_config(fmt=...)` is the explicit way to change branch fmt before `repo.add(".")`.


## 4. Demo for `repo.checkout('BRANCH')` and for `repo.add(".")` to remove deleted files from the data.tsv

We use a temporary branch so the main workflow stays intact.
On this branch we switch to `b*` files first so it is easy to distinguish from `main`.
Because the filename family changes, we update the branch fmt explicitly with `repo.set_config(...)` before using `repo.add(".")`.


In [13]:
repo1.checkout("experiment")


True

In [14]:
%%bash
pushd repo1
rm a*.h5
for f in b{0,0.75}_i{0,30,60,90,120}.h5; do echo "$f" > "$f"; done
echo "Files after switching experiment to b files:"
ls *.h5
popd


~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo/repo1 ~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo
Files after switching experiment to b files:
b0_i0.h5
b0_i120.h5
b0_i30.h5
b0_i60.h5
b0_i90.h5
b0.75_i0.h5
b0.75_i120.h5
b0.75_i30.h5
b0.75_i60.h5
b0.75_i90.h5
~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo


In [15]:
repo1.set_config(fmt="b{a}_i{i}.h5")
repo1.state.config


{'data': [{'fmt': 'b{a}_i{i}.h5', 'encoding': None}], 'remote': None}

In [16]:
%%bash
pushd repo1
echo "config.yml after repo.set_config:"
cat .hm/config.yml
popd


~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo/repo1 ~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo
config.yml after repo.set_config:
data:
- fmt: b{a}_i{i}.h5
  encoding: null
remote: null
~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo


In [17]:
%%bash
pushd repo1
rm b0.75_i{0,30,60,90,120}.h5
echo "Files after deleting some tracked b files:"
ls *.h5
popd


~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo/repo1 ~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo
Files after deleting some tracked b files:
b0_i0.h5
b0_i120.h5
b0_i30.h5
b0_i60.h5
b0_i90.h5
~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo


In [18]:
repo1.add(".")


,path,a,i
0,b0_i0.h5,0.0,0.0
1,b0_i120.h5,0.0,120.0
2,b0_i30.h5,0.0,30.0
3,b0_i60.h5,0.0,60.0
4,b0_i90.h5,0.0,90.0


In [19]:
%%bash
echo "Manifest after repo.add dot:"
cat repo1/.hm/data.tsv


Manifest after repo.add dot:
sha1	a	i
56676e3e5629923617f8c9e542bd1c2ccaf8fa3a	0	0
683e82f624b175c075ac3ea0cca56e94cd7422f8	0	120
cfdbae05e3afc29354d948a32263f7ac1ed5850b	0	30
3a947fb1280b5bca62615d6c7543b88080588e7a	0	60
1711d3b47eac6ae479b6f0f240f83f41b393ebb4	0	90


In [20]:
repo1.status()


{'branch': 'experiment',
 'staged': {'added': ['b0_i0.h5',
   'b0_i120.h5',
   'b0_i30.h5',
   'b0_i60.h5',
   'b0_i90.h5'],
  'modified': [],
  'deleted': ['a0.75_i0.h5',
   'a0.75_i120.h5',
   'a0.75_i30.h5',
   'a0.75_i60.h5',
   'a0.75_i90.h5',
   'a0_i0.h5',
   'a0_i120.h5',
   'a0_i30.h5',
   'a0_i60.h5',
   'a0_i90.h5']},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

In [21]:
repo1.commit("sync manifest with hallmark add dot")


True

In [22]:
repo1.checkout("main")


True

In [23]:
%%bash
echo "Files after checkout back to main:"
ls repo1/*.h5


Files after checkout back to main:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5


## 5. Uncommitted changes block checkout


In [24]:
repo1.checkout("experiment")


True

In [25]:
%%bash
echo "Files after checkout experiment:"
ls repo1/*.h5
echo ""
echo "experiment data.tsv:"
cat repo1/.hm/data.tsv


Files after checkout experiment:
repo1/b0_i0.h5
repo1/b0_i120.h5
repo1/b0_i30.h5
repo1/b0_i60.h5
repo1/b0_i90.h5

experiment data.tsv:
sha1	a	i
56676e3e5629923617f8c9e542bd1c2ccaf8fa3a	0	0
683e82f624b175c075ac3ea0cca56e94cd7422f8	0	120
cfdbae05e3afc29354d948a32263f7ac1ed5850b	0	30
3a947fb1280b5bca62615d6c7543b88080588e7a	0	60
1711d3b47eac6ae479b6f0f240f83f41b393ebb4	0	90


In [26]:
%%bash
pushd repo1
echo "b2_i15.h5" > b2_i15.h5
echo "Files after creating b2_i15.h5:"
ls *.h5
popd


~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo/repo1 ~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo
Files after creating b2_i15.h5:
b0_i0.h5
b0_i120.h5
b0_i30.h5
b0_i60.h5
b0_i90.h5
b2_i15.h5
~/Library/CloudStorage/OneDrive-Personal/imp_files/rohin_school/university_of_arizona/eht_software/hallmark/demo


In [27]:
repo1.add(".")


,path,a,i
0,b0_i0.h5,0.0,0.0
1,b0_i120.h5,0.0,120.0
2,b0_i30.h5,0.0,30.0
3,b0_i60.h5,0.0,60.0
4,b0_i90.h5,0.0,90.0
5,b2_i15.h5,2.0,15.0


### Dirty status


In [28]:
repo1.status()


{'branch': 'experiment',
 'staged': {'added': ['b2_i15.h5'], 'modified': [], 'deleted': []},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

In [29]:
repo1.checkout("main")

CheckoutError: you have uncommitted hallmark state changes — commit them before checkout

To fix this, commit the staged changes first or discard them.


## 6. Clone a hallmark repository

This clone demo matches the CLI notebook: it clones the hallmark repo, then downloads the configured remote data files. If you rerun the clone cell, the notebook catches `DestinationExistsError` and prints only Hallmark's fatal message.


In [30]:
clone_url = "https://github.com/l6a/hallmark-demo-repo.hm.git"
clone_path = Path("hallmark-demo-clone")

try:
    cloned_repo = Repo.clone(clone_url, clone_path)
    print(f'Successfully cloned to "{clone_path}"')
    print("Downloading remote data files...")
    results = download_remote_data(cloned_repo, clone_path, show_progress=True)
    if results["failed"] == 0:
        mb_total = results["total_bytes"] / (1024 * 1024)
        print(f'Successfully downloaded {results["succeeded"]} files ({mb_total:.1f} MB)')
    else:
        print(
            "Download completed with errors: "
            f'{results["succeeded"]} succeeded, {results["failed"]} failed'
        )
        for error in results["errors"]:
            print(f'  - {error}')
except DestinationExistsError as exc:
    print(exc)


Successfully cloned to "hallmark-demo-clone"


100%|██████████| 8/8 [00:00<00:00, 20.56file/s]

Successfully downloaded 8 files (4.3 MB)


In [33]:
cloned_repo = Repo(clone_path)
print(f'dot-hallmark repo: "demo/{Path(cloned_repo.dothm.path).relative_to(Path.cwd())}"')
print(f'hallmark worktree: "demo/{Path(cloned_repo.worktree).relative_to(Path.cwd())}"')


dot-hallmark repo: "demo/hallmark-demo-clone/.hm"
hallmark worktree: "demo/hallmark-demo-clone"


In [34]:
%%bash
echo "=== cloned config.yml ==="
cat hallmark-demo-clone/.hm/config.yml


=== cloned config.yml ===
data:
  - fmt: '{release}_{source}_{year}_{doy:03d}_{band}_{pipeline}_{step}_{type}.uvfits'
remote:
  name: origin
  url: https://data.cyverse.org/dav-anon/iplant/commons/cyverse_curated/EHTC_FirstM87Results_Apr2019/uvfits/